In [1]:
import numpy as np, pandas as pd, time, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

d = np.load("../data/processed/fd001_windows.npz")
X_tr, y_tr, X_val, y_val = d["X_tr"], d["y_tr"], d["X_val"], d["y_val"]
X_test, y_test = d["X_test"], np.minimum(d["y_test"], 125)
n_feat, W = X_tr.shape[2], X_tr.shape[1]

def rmse(y, p):
    return float(np.sqrt(np.mean((y - p) ** 2)))

def nasa_score(y, p):
    e = p - y
    return float(np.sum(np.where(e < 0, np.exp(-e / 13) - 1, np.exp(e / 10) - 1)))

train_dl = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)), batch_size=256, shuffle=True)
results = {}
print(device, X_tr.shape)

cpu (14241, 30, 15)


In [2]:
class LSTMNet(nn.Module):
    def __init__(self, n_feat, hidden=64, layers=2, drop=0.2):
        super().__init__()
        self.rnn = nn.LSTM(n_feat, hidden, layers, batch_first=True, dropout=drop)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1]).squeeze(-1)

class GRUNet(nn.Module):
    def __init__(self, n_feat, hidden=64, layers=2, drop=0.2):
        super().__init__()
        self.rnn = nn.GRU(n_feat, hidden, layers, batch_first=True, dropout=drop)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1]).squeeze(-1)

class CNN1D(nn.Module):
    def __init__(self, n_feat, W, drop=0.2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_feat, 64, 5, padding=2), nn.ReLU(),
            nn.Conv1d(64, 64, 5, padding=2), nn.ReLU(),
            nn.Flatten())
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(64 * W, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x):
        return self.head(self.conv(x.transpose(1, 2))).squeeze(-1)   # conv wants (batch, features, time)

In [3]:
CAP = 125.0
train_dl = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr / CAP)), batch_size=256, shuffle=True)

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X).to(device)).cpu().numpy() * CAP

def train_model(model, name, epochs=80, lr=1e-3, patience=10, min_epochs=15):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best, best_state, wait = 1e9, None, 0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        v = rmse(y_val, predict(model, X_val))
        if v < best:
            best, wait = v, 0
            best_state = {k: t.clone() for k, t in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience and ep >= min_epochs:
                break
        if ep % 5 == 0:
            print(f"{name} epoch {ep + 1}: val RMSE {v:.3f}")
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), f"../results/{name}.pt")
    p = predict(model, X_test)
    return {"val_RMSE": best, "test_RMSE": rmse(y_test, p), "test_NASA": nasa_score(y_test, p),
            "test_RMSE_raw": rmse(d["y_test"], p), "test_NASA_raw": nasa_score(d["y_test"], p),
            "time_s": time.time() - t0}

In [4]:
torch.manual_seed(42)
results["LSTM"] = train_model(LSTMNet(n_feat), "LSTM")
print(results["LSTM"])

LSTM epoch 1: val RMSE 33.758
LSTM epoch 6: val RMSE 13.676
LSTM epoch 11: val RMSE 13.730
LSTM epoch 16: val RMSE 13.055
LSTM epoch 21: val RMSE 12.876
LSTM epoch 26: val RMSE 12.389
LSTM epoch 31: val RMSE 12.322
LSTM epoch 36: val RMSE 12.476
LSTM epoch 41: val RMSE 13.232
{'val_RMSE': 11.920276641845703, 'test_RMSE': 14.185336112976074, 'test_NASA': 391.096435546875, 'test_RMSE_raw': 15.354776382446289, 'test_NASA_raw': 423.38775634765625, 'time_s': 223.60853600502014}


In [5]:
torch.manual_seed(42)
results["GRU"] = train_model(GRUNet(n_feat), "GRU")
print(results["GRU"])

GRU epoch 1: val RMSE 20.969
GRU epoch 6: val RMSE 14.599
GRU epoch 11: val RMSE 13.472
GRU epoch 16: val RMSE 12.955
GRU epoch 21: val RMSE 11.915
GRU epoch 26: val RMSE 12.659
GRU epoch 31: val RMSE 12.074
{'val_RMSE': 11.594995498657227, 'test_RMSE': 13.616103172302246, 'test_NASA': 330.50677490234375, 'test_RMSE_raw': 14.880180358886719, 'test_NASA_raw': 365.6521301269531, 'time_s': 254.54757809638977}


In [6]:
torch.manual_seed(42)
results["1D-CNN"] = train_model(CNN1D(n_feat, W), "CNN1D")
print(results["1D-CNN"])

CNN1D epoch 1: val RMSE 19.644
CNN1D epoch 6: val RMSE 12.800
CNN1D epoch 11: val RMSE 12.336
CNN1D epoch 16: val RMSE 12.092
CNN1D epoch 21: val RMSE 12.698
{'val_RMSE': 11.982942581176758, 'test_RMSE': 13.397027969360352, 'test_NASA': 313.49993896484375, 'test_RMSE_raw': 14.563169479370117, 'test_NASA_raw': 344.9320983886719, 'time_s': 51.30483627319336}


In [7]:
base = pd.read_csv("../results/baselines.csv", index_col=0)
table = pd.concat([base, pd.DataFrame(results).T]).round(2)
table.to_csv("../results/metrics_table.csv")
table

,val_RMSE,test_RMSE,test_NASA,time_s,test_RMSE_raw,test_NASA_raw
RandomForest,12.88,12.91,278.27,59.17,14.11,310.99
XGBoost,12.56,12.57,260.91,7.95,13.72,288.56
LSTM,11.92,14.19,391.10,223.61,15.35,423.39
GRU,11.59,13.62,330.51,254.55,14.88,365.65
1D-CNN,11.98,13.40,313.50,51.30,14.56,344.93
